In [ ]:
seed = 42
target = 'recur'

import json
import os
from glob import glob
import random
from tqdm import tqdm
from datetime import datetime

# font_fname = os.environ.get('PROJECT_FONT_PATH', '') #적용할 폰트
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

from src.utils.preprocessing import Preprocessor
from src.models import SupervisedTextDataset, SupervisedPLModel, F1MacroCallback

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from umap import UMAP

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from src.utils.preprocessing import Preprocessor


# 랜덤 시드 설정 (PyTorch Lightning 권장 방식)

# 파이썬 해시 시드
os.environ['PYTHONHASHSEED'] = str(seed)

# 파이썬 random
random.seed(seed)

# numpy, torch, pytorch-lightning는 이미 다른 셀에서 import 되어 있으므로 그대로 사용
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# PyTorch Lightning utility (workers=True로 데이터로더 워커 시드도 설정)
pl.seed_everything(seed, workers=True)

# save_dir_version = 'v002'
# save_dir_prefix = f'CLF-{target}-{inter_fix}'

font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

# data_dir_path = sorted(glob(os.path.join('data','reviewed_labels',f'*_{target}.csv')))
# data_dir_path = sorted(glob(os.path.join('data','reports', f'{target}*')))
# print(data_dir_path)

label_dict = {'negative': 0, 'uncertain': 1, 'positive': 2}
inv_label_dict = {v: k for k, v in label_dict.items()}

target_name_dict = {
    'recur': 'Recurrence',
    'metas': 'Metastasis'
}

negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]


def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


def get_model_predictions(pl_model, data_loader):
    """
    Returns logits, targets, and predicted labels for a given model and dataloader.
    """
    pl_model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pl_model.to(device)
    all_logits = []
    all_targets = []
    for batch in tqdm(data_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch.get('attention_mask', None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
            
        if 'labels' in batch:
            labels = batch['labels'].to(device)
            all_targets.append(labels.cpu().numpy())
        else:
            labels = None
        with torch.no_grad():
            logits = pl_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        all_logits.append(logits.cpu().numpy())
        
    all_logits = np.concatenate(all_logits)
    all_targets = np.concatenate(all_targets) if len(all_targets) > 0 else None
    all_preds = np.argmax(all_logits, axis=1)
    return dict(
        logits = all_logits, 
        targets = all_targets, 
        preds = all_preds
    )


In [ ]:

model_dict = dict(
    recur = 'PubMedBERT-base-uncased-sts-combined',
    metas = 'MedEmbed-base-v0.1',
)
model_dir_path = os.path.join('models')


In [ ]:
## raw text

raw_data_dir = glob(os.path.join('data','reports','*'))
print(raw_data_dir)
df = pd.read_csv(raw_data_dir[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


In [ ]:
target_pred_dict = {}


for target in ['recur', 'metas']:
    
    data_dict = preprocessor.target_filtering(target)
    target_df = pd.concat([check_label(d, k) for k, d in data_dict.items()]).sort_index()
    target_df.shape, target_df.drop_duplicates('검사결과결론내용').shape

    ckpt_dir = glob(os.path.join('logs',f'*{target}*', 'files','*epoch*'))[-1]

    model_path = model_dict[target]

    tokenizer = AutoTokenizer.from_pretrained(os.path.join(model_dir_path, model_path))
    language_model = AutoModel.from_pretrained(
            os.path.join(model_dir_path, model_path),
            # dtype="bfloat16",
        )

        
    # Lightning model
    pl_model = SupervisedPLModel.load_from_checkpoint(
        ckpt_dir,
        encoder=language_model,
        hidden_dim=language_model.config.hidden_size,
        num_classes=len(label_dict),
        # lr=1e-5,
        # gamma=2.0,
        # alpha=alpha
    )

    test_dataset = SupervisedTextDataset(target_df.drop_duplicates('검사결과결론내용'), tokenizer,text_col = '검사결과결론내용')
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)   

    prediction = get_model_predictions(pl_model, test_loader)

    from scipy.special import softmax
    probs = softmax(prediction['logits'], axis=1)
    predicted_probs = probs[np.arange(len(probs)), prediction['preds']]

    # drop_duplicates로 중복제거된 target_df에 각 텍스트별로 매칭된 값들마다 probs랑 predicted_probs 병합
    dedup_target_df = target_df.drop_duplicates('검사결과결론내용').reset_index(drop=True)
    # dedup_target_df의 행 수와 probs/predicted_probs의 행 수가 일치한다고 가정

    # probs는 (N, num_classes), predicted_probs는 (N,)
    dedup_target_df[target] = prediction['preds']
    for k, n in label_dict.items():
        dedup_target_df[target+'_'+k] = probs[:, n]
    dedup_target_df[target+'_prob'] = predicted_probs
    pred_df = pd.merge(
        target_df.drop(columns = ['label']).reset_index(),
        dedup_target_df,
        on = '검사결과결론내용',
        how = 'left'
    )
    pred_df.index = pred_df['index'].values
    pred_df[target] = pred_df[target].map(inv_label_dict)
    full_index = pd.DataFrame({'index': np.arange(0, df.index.max() + 1)})
    filled_df = pd.merge(full_index, pred_df, on='index', how='left')
    filled_df = filled_df.drop(columns = ['index']).set_axis(
        [target_name_dict[target]+i for i in ['_text','_rule','_class','_negative','_uncertain','_positive', '_prob']], axis = 1)

    target_pred_dict[target] = filled_df.copy()

In [ ]:

cancer_type_df = pd.read_excel(os.path.join('data','cancer_type_map.xlsx'))

cancer_dict_path = os.path.join('.', 'cancer_dict.json')
with open(cancer_dict_path, 'r', encoding='utf-8') as f:
    cancer_dict = json.load(f)
cancer_en_dict = {int(k) : v['name_en'] for k, v in cancer_dict.items()}

cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Merged'
cnacer_merged_en_dict = {k:v if v not in cancer_merge_keys else merged_cancer_label for k, v in cancer_en_dict.items() }

In [ ]:
cancer_type_df['암종'] = cancer_type_df['암번호'].fillna(99).astype(int).apply(lambda x: cancer_en_dict[x] if x in cancer_en_dict else 'Missing')
cancer_type_df['병합암종'] = cancer_type_df['암번호'].fillna(99).astype(int).apply(lambda x: cnacer_merged_en_dict[x] if x in cnacer_merged_en_dict else 'Missing')

In [ ]:
df = df.merge(cancer_type_df.drop(columns = ['암번호']), on = ['병원등록번호', '수술일자'], how = 'left')

In [ ]:
df.loc[:, 'Metastasis'] = target_pred_dict['metas']['Metastasis_class'].fillna('negative')
df.loc[:, 'Recurrence'] = target_pred_dict['recur']['Recurrence_class'].fillna('negative')

In [ ]:
save_dir = os.path.join('outputs','predictions')
os.makedirs(save_dir, exist_ok = True)
df.to_excel(os.path.join(save_dir, 'raw_format.xlsx'), index = False)

In [ ]:
detail_df = pd.concat([
    df[['병원등록번호','수술일자','성별','검사나이','암종','병합암종','검사결과결론내용']],
    target_pred_dict['metas'],
    target_pred_dict['recur']
], axis = 1)

In [ ]:
detail_df.to_excel(os.path.join(save_dir, 'prediction_format.xlsx'), index = False)